# Ch 7. 스트리밍과 UX

원본: [AI Assistant Engineering - Part 2 Ch 7](https://desty.github.io/study-ai-assistant-engineering/part2/07-streaming-ux/)

## 이 챕터에서 배우는 것

- **TTFT**(Time to First Token) 가 체감 속도의 핵심인 이유
- SDK의 `stream()` 이벤트 흐름 — delta / start / stop
- **취소 · 타임아웃 · 부분 응답** 처리
- 챗봇 UI 에서 **토큰이 들어오는 대로 렌더** 하는 패턴
- 스트림 중간 에러 · 마크다운 부분 렌더 함정

In [ ]:
# 필수 패키지
!pip install -q anthropic

In [ ]:
# API 키 설정
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

In [ ]:
from anthropic import Anthropic
import time

client = Anthropic()

## 4. 최소 예제 — 10줄 스트림

In [ ]:
with client.messages.stream(  # (1)!
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": "파이썬의 매력을 3줄로 설명해줘"}],
) as stream:
    for text in stream.text_stream:  # (2)!
        print(text, end="", flush=True)  # (3)!

## 5. 실전 튜토리얼

### 5.1 스트림 이벤트 구조

In [ ]:
with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=128,
    messages=[{"role": "user", "content": "안녕"}]) as stream:
    for event in stream:
        print(f"{event.type}", end="")
        if event.type == "content_block_delta":
            print(f" -> {event.delta.text}", end="")
        print()

### 5.2 TTFT · TPS 측정

In [ ]:
t_start = time.perf_counter()
t_first: float | None = None
tokens = 0

with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=512,
    messages=[{"role": "user", "content": "AI Assistant 설계 원칙 5개를 설명해줘"}]) as stream:
    for text in stream.text_stream:
        if t_first is None:
            t_first = time.perf_counter()  # 첫 토큰 순간
        tokens += 1  # 대략적 (실제로는 문자수 ≠ 토큰수)

t_end = time.perf_counter()
ttft = t_first - t_start
total = t_end - t_start
tps = tokens / (t_end - t_first) if t_first else 0

print(f"TTFT={ttft:.2f}s  total={total:.2f}s  ~chars/s={tps:.1f}")

### 5.3 취소와 타임아웃

In [ ]:
import signal

stop = False

def on_sigint(sig, frame):  # (1)!
    global stop
    stop = True

signal.signal(signal.SIGINT, on_sigint)

print("스트리밍 시작 (Ctrl+C로 중단 가능):")
with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=256,
    messages=[{"role": "user", "content": "파이썬 튜토리얼을 써줘"}]) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        if stop:
            print("\n\n[중단됨]")
            break  # (2)!

### 5.4 부분 응답 로깅 & 에러 복구

In [ ]:
buffer = []

try:
    with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=512,
        messages=[{"role": "user", "content": "안녕하세요"}]) as stream:
        for text in stream.text_stream:
            buffer.append(text)  # 받은 즉시 버퍼링
            print(text, end="", flush=True)
        final = stream.get_final_message()
        print(f"\n\n[정상 종료] 최종 메시지 도착")
except Exception as e:
    partial = "".join(buffer)
    print(f"\n\n[에러 발생] {str(e)}")
    print(f"부분 응답 ({len(partial)} 글자) 저장됨")

### 5.5 UI 통합 패턴 — FastAPI SSE

이 섹션은 브라우저/웹 서버가 필요하므로 참고용 코드만 제공합니다.

In [ ]:
# FastAPI SSE 예시 (실행 불가, 참고용)
fastapi_sse_code = """
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from anthropic import Anthropic

app = FastAPI()
client = Anthropic()

@app.get("/stream")
def stream_chat(q: str):
    def gen():
        with client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=512,
            messages=[{"role": "user", "content": q}]) as s:
            for text in s.text_stream:
                yield f"data: {text}\\n\\n"  # SSE 포맷
        yield "event: done\\ndata: ok\\n\\n"
    return StreamingResponse(gen(), media_type="text/event-stream")
"""

print("FastAPI SSE 서버 예시:")
print(fastapi_sse_code)
print("\n브라우저 React 클라이언트는 EventSource 로 이벤트를 받으면 됩니다.")

## 6. 자주 깨지는 포인트

### 실수 1. 스트림 중 JSON을 부분 파싱
- `{"item": "운동` 까지 받은 시점에 `json.loads` → 에러
- 구조화 출력 + 스트리밍은 위험 조합
- 대응: 구조화 출력은 비스트리밍으로

### 실수 2. 스트림 중단 시 부분 응답을 버림
- 네트워크 에러 한 번에 받은 토큰이 사라짐
- 대응: §5.4 패턴 — 받는 즉시 버퍼에 누적

### 실수 3. 마크다운 미완성 렌더
- `**굵게` 까지 온 시점에 렌더 → 이상해짐
- 대응: 스트리밍 중엔 원문 `<pre>`로, 완료 후 마크다운 렌더

### 실수 4. Ctrl+C 후 커넥션이 열려있음
- `break` 만 하고 `with` 밖으로 안 나가면 소켓이 열려있음
- 대응: `with` 블록 전체를 `try/except KeyboardInterrupt`로 감싸기

### 실수 5. 스트림 lifetime을 DB/파일에 안 맞춤
- 각 토큰을 DB에 쓰려 하면 DB 폭발
- 대응: 메모리 버퍼 → 일정 주기로 flush, 또는 종료 시 한 번만

## 7. 운영 시 체크할 점

- [ ] **TTFT · TPS 지표** 매 호출 기록 → p50 / p95 대시보드
- [ ] **최대 응답 시간** 상한 (예: 60초). 초과 시 강제 종료
- [ ] **사용자 취소** 경로 보장
- [ ] **부분 응답 저장** — 실패·취소여도 토큰은 이미 비용 발생
- [ ] **서버 동시 스트림 수** 제한 (커넥션 풀)
- [ ] 브라우저에서 **reconnect 로직** (SSE는 자동, WebSocket은 수동)
- [ ] **구조화 출력과 스트리밍 분리 정책** 명시

---

**다음 챕터** → Ch 8. Tool Calling 기초

지금까지는 **LLM이 텍스트만 반환**. 다음은 LLM이 **외부 함수를 부르게** 하기 — Agent(Part 5)의 기초.